In [1]:
import numpy as np

class Perceptron:
    def __init__(self, learning_rate=0.1, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def activation(self, x):
        return 1 if x >= 0 else 0

    def fit(self, X, y):
        self.weights = np.zeros(X.shape[1])
        self.bias = 0
        for _ in range(self.epochs):
            for xi, yi in zip(X, y):
                y_pred = self.activation(np.dot(xi, self.weights) + self.bias)
                error = yi - y_pred
                self.weights += self.lr * error * xi
                self.bias += self.lr * error

    def predict(self, X):
        return [self.activation(np.dot(xi, self.weights) + self.bias) for xi in X]

In [2]:
X = np.array([[0,0],[0,1],[1,0],[1,1]])

labels = {
    "AND":  [0, 0, 0, 1],
    "OR":   [0, 1, 1, 1],
    "NAND": [1, 1, 1, 0],
    "NOR":  [1, 0, 0, 0],
    "XNOR": [1, 0, 0, 1],
}

# XOR and XNOR are non-linearly separable — perceptron won't converge perfectly
X_not = np.array([[0], [1]])
labels_not = {"NOT": [1, 0]}

In [3]:
def test_gate(name, X, y):
    p = Perceptron(learning_rate=0.1, epochs=100)
    p.fit(X, np.array(y))
    predictions = p.predict(X)
    print(f"\n{'='*30}")
    print(f"  Gate: {name}")
    print(f"{'='*30}")
    print(f"  Weights : {p.weights}")
    print(f"  Bias    : {p.bias}")
    print(f"{'─'*30}")
    print(f"  {'A':<5} {'B':<5} Expected  Predicted")
    print(f"{'─'*30}")
    for xi, yi, pi in zip(X, y, predictions):
        match = "✓" if yi == pi else "✗"
        inputs = "  ".join(str(int(v)) for v in xi)
        print(f"  {inputs:<10} {yi:<10} {pi}  {match}")
    accuracy = sum(yi == pi for yi, pi in zip(y, predictions)) / len(y) * 100
    print(f"{'─'*30}")
    print(f"  Accuracy: {accuracy:.0f}%")

# Test all 2-input gates
for name, y in labels.items():
    test_gate(name, X, y)

# Test NOT gate (1 input)
test_gate("NOT", X_not, labels_not["NOT"])


  Gate: AND
  Weights : [0.2 0.1]
  Bias    : -0.20000000000000004
──────────────────────────────
  A     B     Expected  Predicted
──────────────────────────────
  0  0       0          0  ✓
  0  1       0          0  ✓
  1  0       0          0  ✓
  1  1       1          1  ✓
──────────────────────────────
  Accuracy: 100%

  Gate: OR
  Weights : [0.1 0.1]
  Bias    : -0.1
──────────────────────────────
  A     B     Expected  Predicted
──────────────────────────────
  0  0       0          0  ✓
  0  1       1          1  ✓
  1  0       1          1  ✓
  1  1       1          1  ✓
──────────────────────────────
  Accuracy: 100%

  Gate: NAND
  Weights : [-0.2 -0.1]
  Bias    : 0.2
──────────────────────────────
  A     B     Expected  Predicted
──────────────────────────────
  0  0       1          1  ✓
  0  1       1          1  ✓
  1  0       1          1  ✓
  1  1       0          0  ✓
──────────────────────────────
  Accuracy: 100%

  Gate: NOR
  Weights : [-0.1 -0.1]
  Bias    

In [4]:
print("\n" + "="*35)
print("  XOR via 2-Layer Perceptron Network")
print("="*35)

# XOR = (A NAND B) AND (A OR B)
p_nand = Perceptron(); p_nand.fit(X, np.array([1,1,1,0]))
p_or   = Perceptron(); p_or.fit(X, np.array([0,1,1,1]))
p_and  = Perceptron(); p_and.fit(X, np.array([0,0,0,1]))

expected_xor = [0, 1, 1, 0]

print(f"\n  {'A':<5} {'B':<5} Expected  Predicted")
print(f"{'─'*35}")
for xi, yi in zip(X, expected_xor):
    nand_out = p_nand.predict([xi])[0]
    or_out   = p_or.predict([xi])[0]
    xor_out  = p_and.predict([[nand_out, or_out]])[0]
    match = "✓" if yi == xor_out else "✗"
    print(f"  {int(xi[0]):<5} {int(xi[1]):<5} {yi:<10} {xor_out}  {match}")


  XOR via 2-Layer Perceptron Network

  A     B     Expected  Predicted
───────────────────────────────────
  0     0     0          0  ✓
  0     1     1          1  ✓
  1     0     1          1  ✓
  1     1     0          0  ✓
